In [12]:
import torch 
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer 
from peft import get_peft_model, LoraConfig
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


The CorDA tuning step looked at how to tune a CorDA model efficiently using hugging face's high level tuning library and training loops. In this notebook, I will demonstrate what happens under the hood when tuning a smaller model -- for testing purposes -- and as well as the effectiveness of adapter tuning on downstream tasks. 

### Load the gpt-neo-125M model 

In [13]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125m")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125m")
print(model)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [14]:
GPT_NEO_BLOCK = model.transformer.h
attention_block = []

for gpt_block in GPT_NEO_BLOCK:
    attention_block.append(gpt_block.attn.attention)

print(attention_block[0])

GPTNeoSelfAttention(
  (attn_dropout): Dropout(p=0.0, inplace=False)
  (resid_dropout): Dropout(p=0.0, inplace=False)
  (k_proj): Linear(in_features=768, out_features=768, bias=False)
  (v_proj): Linear(in_features=768, out_features=768, bias=False)
  (q_proj): Linear(in_features=768, out_features=768, bias=False)
  (out_proj): Linear(in_features=768, out_features=768, bias=True)
)


In [15]:
#We'll use an example attention head to demonstrate the Linear layers we're performing Lora on 
attention_1 = attention_block[0]
k_proj = attention_1.k_proj
## Print the raw tensor of the K proj matrix and confirm it's shape 
print(k_proj.weight.shape)
print(k_proj.weight.dtype)

torch.Size([768, 768])
torch.float32


### Save model's old params locally

We'll save the params of each block's q, k, v projections 

In [16]:
import torch as pt
import os

EXPERIMENT_NUMBER = 0
proj_modules = ["k_proj", "q_proj", "v_proj"]

for idx, attention in enumerate(attention_block):
    block_dir = f"../model/lora/base_model_attention_proj_weights/block_{idx}"
    os.makedirs(block_dir, exist_ok=True)
    pt.save(attention.k_proj.weight.detach().cpu(), f"{block_dir}/k_proj.pt")
    pt.save(attention.q_proj.weight.detach().cpu(), f"{block_dir}/q_proj.pt")
    pt.save(attention.v_proj.weight.detach().cpu(), f"{block_dir}/v_proj.pt")


### Define LoRa Initializer

In [17]:
class LoRaHelper(nn.Module):
    def __init__(self, base_layer: nn.Linear, alpha:int , in_dims: int = 768, r:int = 4 ):
        super().__init__()
        
        self.base = base_layer
        self.base.weight.requires_grad = False
        
        #If model contains bias matrices  
        if self.base.bias is not None:
            self.base.bias.requires_grad = False
            
        in_dims = self.base.in_features
        out_dims = self.base.out_features
            
        self.alpha = alpha 
        self.r = r 
        self.scaling = self.alpha / self.r
        
        # Up projection layer: r x k
        self.init_A(r = r , k = in_dims)
        # Down projection layer: d x r 
        self.init_B(r=r, d= out_dims)
        
        #.Parameter subclasses the tensor class and tell us that these tensor are learnable params inside of the nn.Module
        self.A = nn.Parameter(self.A)
        self.B = nn.Parameter(self.B)
         
    def forward(self, x):
        result = self.base(x)
        
        down_proj = x @ self.A.T
        
        #up_proj = BAx
        up_proj = down_proj @ self.B.T
        
        h = result + self.scaling * up_proj 
        return h 
        
    # A has dims R ^ {r * k }
    def init_A(self, r: int, k:int ):
        self.A = torch.empty(r, k)
        self.A = torch.nn.init.normal_(self.A, mean=0.0, std=0.02)
        
    def init_B(self, r:int, d:int):
        self.B = torch.empty(d,r)
        

### Define Hyper params 

In [18]:
EPOCHS = 5 
alpha = 0.02
lr = 0.01
batch_size = 2 

### Load the dataaset 

In [19]:
# from utils import dataset

# # ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")

# ds = dataset.getRawDataset("imdb", "train")
# # ds = ds.with_format("torch")  # keep it as Python objects for preprocessing
# dataloader = DataLoader(ds, batch_size=5, shuffle=True)
# print(ds)

# batch = {
#     'text': [ds[i]['text'] for i in range(5)],
#     'label': [ds[i]['label'] for i in range(5)]
# }

# cleaned = dataset.preprocess(batch, "imdb")
# print("CLEANED\n", cleaned)

In [ ]:
from utils import dataset
from importlib import reload

# dataset = reload(dataset)
batch_size = 15
clean = dataset.getDatasetBatch(batch_size, "imdb")

for i, text in enumerate(clean['text']):
    print(text)

Loaded imdb (train): 25000
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself. The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men. What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between

### Freeze the models learnable parameters

In [21]:
for param in model.parameters():
    param.requires_grad = False

### Quick Sanity Check for param freezing 

In [22]:
for i in model.parameters():
    if i.requires_grad == True:
        print("There is an error in grad freezing")

### Initialize the LoRa Helper Class for each linear projection head

In [23]:
for attention_head in attention_block:
    attention_head.k_proj = LoRaHelper(
        ############################### there was a .base here 
        base_layer= attention_head.k_proj,
        alpha=16
    )

### Create tuning loop

In [26]:
from utils import dataset

tokenizer.pad_token = tokenizer.eos_token

# To Optimize training only use the optimizer on trainable params in the network
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad]
    ,lr=1e-4)

loss_fn = torch.nn.CrossEntropyLoss()

datasets= ['imdb']

for dataset_name in datasets:
    
    ds = dataset.getRawDataset(dataset_name, "train")
    ds = ds.with_format("torch")
    dataloader = DataLoader(ds,batch_size=batch_size, shuffle=True, drop_last=True) 

    for epoch in range(EPOCHS):    
        model.train()

        for step, batch in enumerate(tqdm(dataloader)):      

            cleaned = dataset.preprocess(batch, dataset_name)    

            # here since we are under IMDB we use cleaned["text"]
            tokenized_batch = tokenizer(cleaned['text'], padding=True,truncation=True, return_tensors="pt", max_length=96).to(model.device)
            input_ids = tokenized_batch["input_ids"].to(model.device)
            attention_mask = tokenized_batch["attention_mask"].to(model.device)
            
            # This is the forward pass
            preds = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids) 
            # logits = preds["logits"]
            # loss = preds.loss
            
            # # zero out the gradients 

            # optimizer.zero_grad()
            # #Run back prop once for the step 
            # # loss.backward()
            # #Use the defined optimizer to update the grads accordingly 
            # # Tells the optimizer to take a step in gradient descent and update the grads with the provided learning rate
            # optimizer.step()
            
            # if step % 50 == 0:
            #     print(f"epoch {epoch} step {step} loss: {loss.item()}" )
            
        model.eval()
        # for step, batch in enumerate(tqdm(dataloader)): 
        
        

Loaded imdb (train): 25000


  3%|▎         | 128/5000 [00:46<29:34,  2.74it/s]


KeyboardInterrupt: 